# Cross-task DPS-A aggregation for AGNews + TREC + Yahoo

Qwen3-8B-IT variant with optional positive average matched-preactivation filtering and pairwise feature-overlap analysis.

Run the three task notebooks first. They save per-feature summaries into `./dps_positionwise_wrapped_outputs/{agnews,trec,yahoo}/`. This notebook loads those summaries and counts Qwen3-8B-IT SAE features satisfying DPS-A conditions jointly across all three tasks.


In [ ]:
import json
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

PLOT_FONT = "Times New Roman, Times, serif"
OUTPUT_ROOT = Path("./dps_positionwise_wrapped_outputs")
TASK_KEYS = ["agnews", "trec", "yahoo"]
TASK_DISPLAY_ORDER = ["AGNews", "TREC", "Yahoo"]
POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP = 200

# Expected label counts for the three-task DPS definition: 4 + 6 + 10 = 20.
EXPECTED_NUM_LABELS_BY_TASK = {
    "agnews": 4,
    "trec": 6,
    "yahoo": 10,
}
EXPECTED_TOTAL_LABEL_CONDITIONS = sum(EXPECTED_NUM_LABELS_BY_TASK.values())

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("Expected total DPS-A label conditions:", EXPECTED_TOTAL_LABEL_CONDITIONS)


In [ ]:
def load_task_scores(task_key: str) -> pd.DataFrame:
    path = OUTPUT_ROOT / task_key / "all_feature_scores_latest.csv.gz"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run the {task_key} task notebook first, including the final save cell."
        )
    df = pd.read_csv(path)
    df["task_key"] = df["task_key"].astype(str)
    df["feature_idx"] = df["feature_idx"].astype(int)
    df["varied_position"] = df["varied_position"].astype(int)
    return df


task_dfs = []
for task_key in TASK_KEYS:
    df = load_task_scores(task_key)
    task_dfs.append(df)
    print(task_key, df.shape, "positions:", sorted(df["varied_position"].unique().tolist()), "num_labels:", sorted(df["num_labels"].unique().tolist()))

all_task_scores_df = pd.concat(task_dfs, ignore_index=True)
print("Combined rows:", len(all_task_scores_df))
display(all_task_scores_df.head())


In [ ]:
# Keep only positions that are present in all three task outputs.
positions_by_task = {
    task_key: set(df["varied_position"].astype(int).unique().tolist())
    for task_key, df in zip(TASK_KEYS, task_dfs)
}
common_positions = sorted(set.intersection(*positions_by_task.values()))
print("Common positions:", common_positions)

scores_common_df = all_task_scores_df[
    all_task_scores_df["varied_position"].isin(common_positions)
].copy()

# ============================================================
# Positive matched-preactivation filter
# ============================================================
# This avoids counting features that technically satisfy the DPS-A max condition
# only because the matched condition is the least-negative value.
#
# The default condition is the weighted cross-task average over all label
# conditions:
#   Abar_f = sum_task(num_labels_task * avg_matched_preact_task) / sum_task(num_labels_task)
# and we require Abar_f > 0 on the discovery split.
#
# You can switch to "all_tasks" if you want the stricter condition that every
# task-level average matched preactivation is positive.
REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION = True
POSITIVE_AVG_MATCHED_PREACTIVATION_MODE = "weighted_all_conditions"  # {"weighted_all_conditions", "all_tasks"}
REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION_ON_HOLDOUT_FOR_STILL_COUNT = True

# Weighted sums used to compute cross-task means over all label conditions.
scores_common_df["discovery_matched_preactivation_weighted_sum"] = (
    scores_common_df["discovery_average_matched_preactivation"].astype(float)
    * scores_common_df["num_labels"].astype(float)
)
scores_common_df["holdout_matched_preactivation_weighted_sum"] = (
    scores_common_df["holdout_average_matched_preactivation"].astype(float)
    * scores_common_df["num_labels"].astype(float)
)

cross_task_feature_df = (
    scores_common_df
    .groupby(["varied_position", "feature_idx"], as_index=False)
    .agg(
        task_count=("task_key", "nunique"),
        total_label_conditions=("num_labels", "sum"),
        discovery_dps_a_count_total=("discovery_dps_a_count", "sum"),
        holdout_dps_a_count_total=("holdout_dps_a_count", "sum"),
        discovery_gap_sum_total=("discovery_gap_sum", "sum"),
        holdout_gap_sum_total=("holdout_gap_sum", "sum"),
        discovery_min_gap_across_tasks=("discovery_min_gap", "min"),
        holdout_min_gap_across_tasks=("holdout_min_gap", "min"),

        # Unweighted task means, retained for reference/backward compatibility.
        discovery_mean_matched_preactivation_unweighted=("discovery_average_matched_preactivation", "mean"),
        holdout_mean_matched_preactivation_unweighted=("holdout_average_matched_preactivation", "mean"),

        # Min across tasks, useful for a stricter positivity option.
        discovery_min_matched_preactivation_across_tasks=("discovery_average_matched_preactivation", "min"),
        holdout_min_matched_preactivation_across_tasks=("holdout_average_matched_preactivation", "min"),

        # Weighted sums, where each task contributes in proportion to its number of labels.
        discovery_matched_preactivation_weighted_sum=("discovery_matched_preactivation_weighted_sum", "sum"),
        holdout_matched_preactivation_weighted_sum=("holdout_matched_preactivation_weighted_sum", "sum"),
    )
)

cross_task_feature_df["has_all_tasks"] = cross_task_feature_df["task_count"].eq(len(TASK_KEYS))
cross_task_feature_df["has_expected_label_total"] = cross_task_feature_df["total_label_conditions"].eq(
    EXPECTED_TOTAL_LABEL_CONDITIONS
)

cross_task_feature_df["discovery_mean_gap_cross_task"] = (
    cross_task_feature_df["discovery_gap_sum_total"]
    / cross_task_feature_df["total_label_conditions"]
)
cross_task_feature_df["holdout_mean_gap_cross_task"] = (
    cross_task_feature_df["holdout_gap_sum_total"]
    / cross_task_feature_df["total_label_conditions"]
)

# Weighted cross-task average matched preactivation across all label conditions.
cross_task_feature_df["discovery_mean_matched_preactivation"] = (
    cross_task_feature_df["discovery_matched_preactivation_weighted_sum"]
    / cross_task_feature_df["total_label_conditions"]
)
cross_task_feature_df["holdout_mean_matched_preactivation"] = (
    cross_task_feature_df["holdout_matched_preactivation_weighted_sum"]
    / cross_task_feature_df["total_label_conditions"]
)

cross_task_feature_df["satisfies_all20_discovery"] = (
    cross_task_feature_df["has_all_tasks"]
    & cross_task_feature_df["has_expected_label_total"]
    & cross_task_feature_df["discovery_dps_a_count_total"].eq(cross_task_feature_df["total_label_conditions"])
)
cross_task_feature_df["satisfies_all20_holdout"] = (
    cross_task_feature_df["has_all_tasks"]
    & cross_task_feature_df["has_expected_label_total"]
    & cross_task_feature_df["holdout_dps_a_count_total"].eq(cross_task_feature_df["total_label_conditions"])
)

if POSITIVE_AVG_MATCHED_PREACTIVATION_MODE == "weighted_all_conditions":
    cross_task_feature_df["positive_avg_matched_preactivation_discovery"] = (
        cross_task_feature_df["discovery_mean_matched_preactivation"] > 0
    )
    cross_task_feature_df["positive_avg_matched_preactivation_holdout"] = (
        cross_task_feature_df["holdout_mean_matched_preactivation"] > 0
    )
elif POSITIVE_AVG_MATCHED_PREACTIVATION_MODE == "all_tasks":
    cross_task_feature_df["positive_avg_matched_preactivation_discovery"] = (
        cross_task_feature_df["discovery_min_matched_preactivation_across_tasks"] > 0
    )
    cross_task_feature_df["positive_avg_matched_preactivation_holdout"] = (
        cross_task_feature_df["holdout_min_matched_preactivation_across_tasks"] > 0
    )
else:
    raise ValueError(
        "POSITIVE_AVG_MATCHED_PREACTIVATION_MODE must be "
        "'weighted_all_conditions' or 'all_tasks'."
    )

cross_task_feature_df["satisfies_all20_positive_discovery"] = (
    cross_task_feature_df["satisfies_all20_discovery"]
    & (
        cross_task_feature_df["positive_avg_matched_preactivation_discovery"]
        if REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION
        else True
    )
)

cross_task_feature_df["satisfies_all20_positive_holdout"] = (
    cross_task_feature_df["satisfies_all20_holdout"]
    & (
        cross_task_feature_df["positive_avg_matched_preactivation_holdout"]
        if REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION_ON_HOLDOUT_FOR_STILL_COUNT
        else True
    )
)

print("Cross-task feature table:", cross_task_feature_df.shape)
print("Positive matched-preactivation filter enabled:", REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION)
print("Positive matched-preactivation mode:", POSITIVE_AVG_MATCHED_PREACTIVATION_MODE)
print("Holdout still-count also requires positive matched preactivation:", REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION_ON_HOLDOUT_FOR_STILL_COUNT)
display(cross_task_feature_df.head())

In [ ]:
def _sem(x: Sequence[float]) -> float:
    arr = np.asarray(list(x), dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) <= 1:
        return 0.0
    return float(np.std(arr, ddof=1) / np.sqrt(len(arr)))

summary_rows = []

for position in common_positions:
    sub = cross_task_feature_df[
        cross_task_feature_df["varied_position"].eq(int(position))
    ].copy()

    # Main discovered set: all-20 DPS-A features, optionally additionally requiring
    # positive average matched preactivation on the discovery split.
    discovered = sub[sub["satisfies_all20_positive_discovery"]].copy()

    discovered = discovered.sort_values(
        [
            "discovery_mean_gap_cross_task",
            "discovery_min_gap_across_tasks",
            "discovery_mean_matched_preactivation",
        ],
        ascending=[False, False, False],
    )

    if POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP is None:
        topk = discovered
    else:
        topk = discovered.head(int(POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP))

    summary_rows.append({
        "varied_position": int(position),

        # Positive-filtered main counts.
        "n_discovery_all20_features": int(len(discovered)),
        "n_discovery_all20_still_all20_holdout": int(
            discovered["satisfies_all20_positive_holdout"].sum()
        ) if len(discovered) else 0,
        "n_topk_features": int(len(topk)),

        # Raw diagnostic counts without the positive-preactivation condition.
        "n_discovery_all20_features_without_positive_filter": int(
            sub["satisfies_all20_discovery"].sum()
        ),
        "n_discovery_all20_still_all20_holdout_without_positive_filter": int(
            (sub["satisfies_all20_discovery"] & sub["satisfies_all20_holdout"]).sum()
        ),

        # Preactivation diagnostics.
        "discovery_mean_matched_preactivation_all_discovered": float(
            discovered["discovery_mean_matched_preactivation"].mean()
        ) if len(discovered) else np.nan,
        "holdout_mean_matched_preactivation_all_discovered": float(
            discovered["holdout_mean_matched_preactivation"].mean()
        ) if len(discovered) else np.nan,
        "discovery_mean_matched_preactivation_topk": float(
            topk["discovery_mean_matched_preactivation"].mean()
        ) if len(topk) else np.nan,
        "holdout_mean_matched_preactivation_topk": float(
            topk["holdout_mean_matched_preactivation"].mean()
        ) if len(topk) else np.nan,

        # Gap metrics, evaluated on holdout.
        "holdout_mean_gap_all_discovered": float(
            discovered["holdout_mean_gap_cross_task"].mean()
        ) if len(discovered) else np.nan,
        "holdout_sem_gap_all_discovered": _sem(
            discovered["holdout_mean_gap_cross_task"]
        ) if len(discovered) else np.nan,
        "holdout_mean_gap_topk": float(
            topk["holdout_mean_gap_cross_task"].mean()
        ) if len(topk) else np.nan,
        "holdout_sem_gap_topk": _sem(
            topk["holdout_mean_gap_cross_task"]
        ) if len(topk) else np.nan,

        # Discovery metrics, for sanity checking selection strength.
        "discovery_mean_gap_all_discovered": float(
            discovered["discovery_mean_gap_cross_task"].mean()
        ) if len(discovered) else np.nan,
        "discovery_mean_gap_topk": float(
            topk["discovery_mean_gap_cross_task"].mean()
        ) if len(topk) else np.nan,
    })

cross_task_summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values("varied_position")
    .reset_index(drop=True)
)

print("Cross-task DPS-A summary")
if REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION:
    print(
        "Main counts/gaps use all-20 DPS-A features with positive average "
        f"matched preactivation; mode={POSITIVE_AVG_MATCHED_PREACTIVATION_MODE!r}."
    )
else:
    print("Main counts/gaps use all-20 DPS-A features without a positivity filter.")

display(cross_task_summary_df)

In [ ]:
# ============================================================
# Pairwise overlap of position-specific DPS-A feature sets
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Which feature sets should we compare?
# - main_discovered:
#     Features satisfying the notebook's main discovery criterion:
#       all-20 DPS-A + positive average matched preactivation if enabled.
# - topk_discovered:
#     Top-K subset of main_discovered, ranked by discovery gap.
# - holdout_retained:
#     Main-discovered features that also remain all-20/positive on holdout.
FEATURE_OVERLAP_SET_MODES = [
    "main_discovered",
    "topk_discovered",
    "holdout_retained",
]

INCLUDE_SHARED_FEATURE_IDS_IN_DISPLAY = False
FEATURE_OVERLAP_TOP_K = POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP

POSITION_DISCOVERY_SORT_COLUMNS = [
    "discovery_mean_gap_cross_task",
    "discovery_min_gap_across_tasks",
    "discovery_mean_matched_preactivation",
]

POSITION_DISCOVERY_SORT_ASCENDING = [
    False,
    False,
    False,
]


def get_position_feature_df(position: int, set_mode: str) -> pd.DataFrame:
    """Return the feature dataframe for one position under a given selection mode."""
    position = int(position)

    sub = cross_task_feature_df[
        cross_task_feature_df["varied_position"].eq(position)
    ].copy()

    if set_mode == "main_discovered":
        selected = sub[
            sub["satisfies_all20_positive_discovery"]
        ].copy()

    elif set_mode == "topk_discovered":
        selected = sub[
            sub["satisfies_all20_positive_discovery"]
        ].copy()

        selected = selected.sort_values(
            POSITION_DISCOVERY_SORT_COLUMNS,
            ascending=POSITION_DISCOVERY_SORT_ASCENDING,
        )

        if FEATURE_OVERLAP_TOP_K is not None:
            selected = selected.head(int(FEATURE_OVERLAP_TOP_K))

    elif set_mode == "holdout_retained":
        selected = sub[
            sub["satisfies_all20_positive_discovery"]
            & sub["satisfies_all20_positive_holdout"]
        ].copy()

    else:
        raise ValueError(
            f"Unknown set_mode={set_mode!r}. "
            f"Expected one of {FEATURE_OVERLAP_SET_MODES}."
        )

    selected = selected.sort_values(
        POSITION_DISCOVERY_SORT_COLUMNS,
        ascending=POSITION_DISCOVERY_SORT_ASCENDING,
    ).reset_index(drop=True)

    return selected


def get_position_feature_sets(set_mode: str) -> dict:
    """Return {position: set(feature_idx)} for one selection mode."""
    return {
        int(position): set(
            get_position_feature_df(
                position=int(position),
                set_mode=set_mode,
            )["feature_idx"].astype(int).tolist()
        )
        for position in common_positions
    }


def compute_pairwise_feature_overlap(set_mode: str) -> tuple[pd.DataFrame, dict]:
    """Compute pairwise feature-set overlap for all position pairs."""
    feature_sets = get_position_feature_sets(set_mode)

    rows = []
    shared_feature_ids_by_pair = {}

    for pos_i in common_positions:
        pos_i = int(pos_i)
        features_i = feature_sets[pos_i]

        for pos_j in common_positions:
            pos_j = int(pos_j)
            features_j = feature_sets[pos_j]

            intersection = features_i & features_j
            union = features_i | features_j

            n_i = len(features_i)
            n_j = len(features_j)
            n_intersection = len(intersection)
            n_union = len(union)
            min_size = min(n_i, n_j)

            jaccard = (
                n_intersection / n_union
                if n_union > 0
                else np.nan
            )

            overlap_coefficient = (
                n_intersection / min_size
                if min_size > 0
                else np.nan
            )

            fraction_i_in_j = (
                n_intersection / n_i
                if n_i > 0
                else np.nan
            )

            fraction_j_in_i = (
                n_intersection / n_j
                if n_j > 0
                else np.nan
            )

            pair_key = f"{pos_i}__{pos_j}"
            shared_feature_ids_by_pair[pair_key] = sorted(
                map(int, intersection)
            )

            row = {
                "set_mode": set_mode,
                "position_i": pos_i,
                "position_j": pos_j,
                "n_features_i": n_i,
                "n_features_j": n_j,
                "n_intersection": n_intersection,
                "n_union": n_union,
                "jaccard": jaccard,
                "overlap_coefficient": overlap_coefficient,
                "fraction_i_in_j": fraction_i_in_j,
                "fraction_j_in_i": fraction_j_in_i,
            }

            if INCLUDE_SHARED_FEATURE_IDS_IN_DISPLAY:
                row["shared_feature_ids"] = ",".join(
                    map(str, sorted(intersection))
                )

            rows.append(row)

    return pd.DataFrame(rows), shared_feature_ids_by_pair


overlap_tables = []
overlap_feature_ids = {}

for set_mode in FEATURE_OVERLAP_SET_MODES:
    overlap_df, shared_ids = compute_pairwise_feature_overlap(set_mode)

    overlap_tables.append(overlap_df)
    overlap_feature_ids[set_mode] = shared_ids

feature_overlap_df = pd.concat(
    overlap_tables,
    ignore_index=True,
)

print("Pairwise feature-overlap table:")
display(feature_overlap_df.head(20))

print("Main-discovered feature-set Jaccard overlap:")
display(
    feature_overlap_df[
        feature_overlap_df["set_mode"].eq("main_discovered")
    ]
    .pivot(
        index="position_i",
        columns="position_j",
        values="jaccard",
    )
)

print("Main-discovered feature-set raw intersection counts:")
display(
    feature_overlap_df[
        feature_overlap_df["set_mode"].eq("main_discovered")
    ]
    .pivot(
        index="position_i",
        columns="position_j",
        values="n_intersection",
    )
)

# Save overlap summaries.
AGG_SUMMARY_DIR = OUTPUT_ROOT / "cross_task_aggregation"
AGG_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

feature_overlap_path = AGG_SUMMARY_DIR / "cross_task_position_pair_feature_overlap_latest.csv"
feature_overlap_ids_path = AGG_SUMMARY_DIR / "cross_task_position_pair_shared_feature_ids_latest.json"

feature_overlap_df.to_csv(
    feature_overlap_path,
    index=False,
)

with open(feature_overlap_ids_path, "w") as f:
    json.dump(
        overlap_feature_ids,
        f,
        indent=2,
    )

print("Saved overlap summary:")
print(feature_overlap_path)
print("Saved shared feature-id lists:")
print(feature_overlap_ids_path)


In [ ]:
# ============================================================
# Pairwise overlap visualizations
# ============================================================

import plotly.graph_objects as go
import numpy as np
import pandas as pd

PLOT_FONT = "Times New Roman, Times, serif"


def axis_title(text: str, size: int = 20):
    return dict(
        text=text,
        font=dict(
            size=size,
            family=PLOT_FONT,
        ),
    )


FEATURE_OVERLAP_MODE_LABELS = {
    "main_discovered": (
        "All-20 + positive discovered features"
        if REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION
        else "All-20 discovered features"
    ),
    "topk_discovered": (
        f"Top-{FEATURE_OVERLAP_TOP_K} discovered features"
        if FEATURE_OVERLAP_TOP_K is not None
        else "All discovered features"
    ),
    "holdout_retained": (
        "Discovery features retained on holdout"
    ),
}


FEATURE_OVERLAP_METRIC_LABELS = {
    "n_intersection": "Shared feature count",
    "jaccard": "Jaccard overlap",
    "overlap_coefficient": "Overlap coefficient",
    "fraction_i_in_j": "Fraction of row set in column set",
    "fraction_j_in_i": "Fraction of column set in row set",
}


def _format_heatmap_text(z, metric: str):
    out = []

    for row in z:
        formatted_row = []

        for value in row:
            if pd.isna(value):
                formatted_row.append("")
            elif metric in {"n_intersection", "n_union", "n_features_i", "n_features_j"}:
                formatted_row.append(f"{int(value)}")
            else:
                formatted_row.append(f"{float(value):.3f}")

        out.append(formatted_row)

    return np.asarray(out, dtype=object)


def plot_feature_overlap_heatmap(
    set_mode: str = "main_discovered",
    metric: str = "jaccard",
):
    if set_mode not in FEATURE_OVERLAP_MODE_LABELS:
        raise ValueError(
            f"Unknown set_mode={set_mode!r}. "
            f"Expected one of {list(FEATURE_OVERLAP_MODE_LABELS)}."
        )

    if metric not in FEATURE_OVERLAP_METRIC_LABELS:
        raise ValueError(
            f"Unknown metric={metric!r}. "
            f"Expected one of {list(FEATURE_OVERLAP_METRIC_LABELS)}."
        )

    sub = feature_overlap_df[
        feature_overlap_df["set_mode"].eq(set_mode)
    ].copy()

    matrix_df = (
        sub
        .pivot(
            index="position_i",
            columns="position_j",
            values=metric,
        )
        .reindex(
            index=common_positions,
            columns=common_positions,
        )
    )

    z = matrix_df.to_numpy(dtype=float)
    text = _format_heatmap_text(z, metric=metric)

    fig = go.Figure(data=go.Heatmap(
        z=z,
        x=[str(p) for p in common_positions],
        y=[str(p) for p in common_positions],
        text=text,
        texttemplate="%{text}",
        colorscale="Blues",
        colorbar=dict(
            title=dict(
                text=FEATURE_OVERLAP_METRIC_LABELS[metric],
            ),
        ),
        hovertemplate=(
            "Position i: %{y}<br>"
            "Position j: %{x}<br>"
            f"{FEATURE_OVERLAP_METRIC_LABELS[metric]}: "
            "%{z:.5f}<extra></extra>"
        ),
    ))

    fig.update_layout(
        title=dict(
            text=(
                "Pairwise feature overlap across discovery positions"
                f"<br><sup>{FEATURE_OVERLAP_MODE_LABELS[set_mode]}</sup>"
            ),
            x=0.0,
            xanchor="left",
            font=dict(
                size=24,
                family=PLOT_FONT,
            ),
        ),
        xaxis=dict(
            title=axis_title("Position j", size=20),
            tickfont=dict(
                size=16,
                family=PLOT_FONT,
            ),
        ),
        yaxis=dict(
            title=axis_title("Position i", size=20),
            autorange="reversed",
            tickfont=dict(
                size=16,
                family=PLOT_FONT,
            ),
        ),
        width=760,
        height=680,
        template="plotly_white",
        font=dict(
            family=PLOT_FONT,
            size=14,
        ),
    )

    filename_metric = metric.replace("_", "-")
    filename_mode = set_mode.replace("_", "-")

    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": f"cross_task_feature_overlap_{filename_mode}_{filename_metric}",
            "width": 760,
            "height": 680,
            "scale": 4,
        }
    })

    return fig


# Main recommended visualizations.
plot_feature_overlap_heatmap(
    set_mode="main_discovered",
    metric="jaccard",
)

plot_feature_overlap_heatmap(
    set_mode="main_discovered",
    metric="n_intersection",
)

# Optional: compare equal-sized top-K groups.
plot_feature_overlap_heatmap(
    set_mode="topk_discovered",
    metric="jaccard",
)

plot_feature_overlap_heatmap(
    set_mode="topk_discovered",
    metric="n_intersection",
)


In [ ]:
def axis_title(text: str, size: int = 20):
    return dict(text=text, font=dict(size=size, family=PLOT_FONT))


COUNT_TRACE_DISCOVERY_NAME = (
    "Discovered all-20 + positive features"
    if REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION
    else "Discovered all-20 features"
)
COUNT_TRACE_HOLDOUT_NAME = (
    "Still all-20 + positive on holdout"
    if REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION_ON_HOLDOUT_FOR_STILL_COUNT
    else "Still all-20 on holdout"
)
GAP_TRACE_ALL_NAME = (
    "All discovered all-20 + positive features"
    if REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION
    else "All discovered all-20 features"
)


def plot_cross_task_dps_counts():
    sub = cross_task_summary_df.sort_values("varied_position")
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=sub["varied_position"],
        y=sub["n_discovery_all20_features"],
        name=COUNT_TRACE_DISCOVERY_NAME,
    ))
    fig.add_trace(go.Bar(
        x=sub["varied_position"],
        y=sub["n_discovery_all20_still_all20_holdout"],
        name=COUNT_TRACE_HOLDOUT_NAME,
    ))
    if 1 in set(sub["varied_position"].astype(int)):
        fig.add_vrect(
            x0=0.5,
            x1=1.5,
            fillcolor="gray",
            opacity=0.10,
            line_width=0,
            annotation_text="first demo",
            annotation_position="top left",
        )
    fig.update_layout(
        title=dict(
            text=(
                "Cross-task positive DPS-A feature count by varied position"
                if REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION
                else "Cross-task DPS-A feature count by varied position"
            ),
            x=0.0,
            xanchor="left",
            font=dict(size=26, family=PLOT_FONT),
        ),
        xaxis=dict(
            title=axis_title("Discovery / varied demonstration position", size=20),
            tickmode="array",
            tickvals=common_positions,
            ticktext=[str(p) for p in common_positions],
            tickfont=dict(size=16, family=PLOT_FONT),
        ),
        yaxis=dict(
            title=axis_title("Number of Qwen3-8B SAE features", size=20),
            tickfont=dict(size=16, family=PLOT_FONT),
        ),
        barmode="group",
        width=950,
        height=560,
        template="plotly_white",
        font=dict(family=PLOT_FONT, size=14),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
    )
    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "cross_task_dps_a_positive_counts_by_position" if REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION else "cross_task_dps_a_counts_by_position",
            "width": 950,
            "height": 560,
            "scale": 4,
        }
    })
    return fig


def plot_cross_task_dps_gap():
    sub = cross_task_summary_df.sort_values("varied_position")
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=sub["varied_position"],
        y=sub["holdout_mean_gap_all_discovered"],
        error_y=dict(type="data", array=sub["holdout_sem_gap_all_discovered"], visible=True, width=4),
        mode="lines+markers",
        name=GAP_TRACE_ALL_NAME,
    ))
    if POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP is not None:
        fig.add_trace(go.Scatter(
            x=sub["varied_position"],
            y=sub["holdout_mean_gap_topk"],
            error_y=dict(type="data", array=sub["holdout_sem_gap_topk"], visible=True, width=4),
            mode="lines+markers",
            name=f"Top-{POSITION_TREND_TOP_K_FOR_EQUALIZED_GAP} per position",
        ))
    if 1 in set(sub["varied_position"].astype(int)):
        fig.add_vrect(
            x0=0.5,
            x1=1.5,
            fillcolor="gray",
            opacity=0.10,
            line_width=0,
            annotation_text="first demo",
            annotation_position="top left",
        )
    fig.update_layout(
        title=dict(
            text=(
                "Cross-task positive DPS-A feature groups: held-out matched gap by discovery position"
                if REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION
                else "Cross-task DPS-A feature groups: held-out matched gap by discovery position"
            ),
            x=0.0,
            xanchor="left",
            font=dict(size=26, family=PLOT_FONT),
        ),
        xaxis=dict(
            title=axis_title("Discovery / varied demonstration position", size=20),
            tickmode="array",
            tickvals=common_positions,
            ticktext=[str(p) for p in common_positions],
            tickfont=dict(size=16, family=PLOT_FONT),
        ),
        yaxis=dict(
            title=axis_title("Held-out matched − best non-matched preactivation", size=20),
            zeroline=True,
            tickfont=dict(size=16, family=PLOT_FONT),
        ),
        width=950,
        height=560,
        template="plotly_white",
        font=dict(family=PLOT_FONT, size=14),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
    )
    fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "cross_task_dps_a_positive_gap_by_position" if REQUIRE_POSITIVE_AVG_MATCHED_PREACTIVATION else "cross_task_dps_a_gap_by_position",
            "width": 950,
            "height": 560,
            "scale": 4,
        }
    })
    return fig


plot_cross_task_dps_counts()
plot_cross_task_dps_gap()

In [ ]:
# Optional: save the combined aggregation tables.
AGG_SUMMARY_DIR = OUTPUT_ROOT / "cross_task_aggregation"
AGG_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

cross_task_feature_path = AGG_SUMMARY_DIR / "cross_task_feature_scores_latest.csv.gz"
cross_task_summary_path = AGG_SUMMARY_DIR / "cross_task_position_summary_latest.csv"

cross_task_feature_df.to_csv(cross_task_feature_path, index=False, compression="gzip")
cross_task_summary_df.to_csv(cross_task_summary_path, index=False)

print("Saved:")
print(cross_task_feature_path)
print(cross_task_summary_path)
